# Pneumonia Detection Web Application

## Overview

This notebook prepares the trained deep learning model for deployment as an interactive web application.

The deployment pipeline includes:

- Loading the trained ResNet18 model
- Creating reusable prediction functions
- Integrating Grad-CAM explainability
- Preparing the Streamlit application
- Saving all deployment files

The final application allows users to upload a chest X-ray image, receive a pneumonia prediction, view the prediction confidence, and visualize the Grad-CAM explanation highlighting the regions that influenced the model's decision.

In [1]:
import os
import cv2
import torch
import numpy as np

from pathlib import Path
from PIL import Image

import torch.nn as nn

from torchvision import transforms
from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/pneumonia_project")

MODEL_DIR = PROJECT_DIR / "models"
SRC_DIR = PROJECT_DIR / "src"

BEST_MODEL = MODEL_DIR / "best_model.pth"

SRC_DIR.mkdir(exist_ok=True)

print("Project Ready")

Project Ready


## Image Preprocessing

Uploaded chest X-ray images must undergo the same preprocessing steps used during model training.

Each image is resized to 224 × 224 pixels, converted into a tensor, and normalized using ImageNet statistics before being passed to the trained neural network.

In [4]:
IMG_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [5]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

## Loading the Trained Model

The ResNet18 architecture is recreated and the trained weights saved during the training phase are loaded.

The model is switched to evaluation mode to ensure consistent behavior during inference. In evaluation mode, layers such as Batch Normalization and Dropout operate deterministically, producing reliable predictions for uploaded chest X-ray images.

In [7]:
# ==========================================
# Build ResNet18 Architecture
# ==========================================

from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

num_features = model.fc.in_features

model.fc = nn.Linear(
    in_features=num_features,
    out_features=2
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


In [9]:
# ==========================================
# Configure Device
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [10]:
# ==========================================
# Load Trained Model
# ==========================================

model.load_state_dict(
    torch.load(BEST_MODEL, map_location=device)
)

model = model.to(device)

model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [11]:
print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


## Image Prediction Function

A reusable prediction function is implemented to perform inference on a single chest X-ray image.

The function applies the same preprocessing pipeline used during training, passes the image through the trained model, computes prediction probabilities using the softmax function, and returns the predicted class along with its confidence score.

This function will be reused by the deployment application to classify user-uploaded chest X-ray images.

In [12]:
# ==========================================
# Label Mapping
# ==========================================

label_map = {
    0: "NORMAL",
    1: "PNEUMONIA"
}

In [13]:
import torch.nn.functional as F

# ==========================================
# Prediction Function
# ==========================================

def predict_image(image_path):

    # Open image
    image = Image.open(image_path).convert("RGB")

    # Preprocess
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Prediction
    model.eval()

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = F.softmax(outputs, dim=1)

        confidence, prediction = torch.max(probabilities, dim=1)

    prediction = prediction.item()
    confidence = confidence.item()

    return (
        label_map[prediction],
        confidence
    )

In [15]:
from pathlib import Path

TEST_DIR = PROJECT_DIR / "data" / "chest_xray" / "test"

for item in TEST_DIR.iterdir():
    print(item)

/content/drive/MyDrive/pneumonia_project/data/chest_xray/test/NORMAL
/content/drive/MyDrive/pneumonia_project/data/chest_xray/test/PNEUMONIA


In [16]:
from pathlib import Path

normal_images = list((PROJECT_DIR / "data" / "chest_xray" / "test" / "NORMAL").glob("*"))
pneumonia_images = list((PROJECT_DIR / "data" / "chest_xray" / "test" / "PNEUMONIA").glob("*"))

print("NORMAL:", normal_images[0])
print("PNEUMONIA:", pneumonia_images[0])

NORMAL: /content/drive/MyDrive/pneumonia_project/data/chest_xray/test/NORMAL/IM-0001-0001.jpeg
PNEUMONIA: /content/drive/MyDrive/pneumonia_project/data/chest_xray/test/PNEUMONIA/person100_bacteria_475.jpeg


In [17]:
sample_image = normal_images[0]

prediction, confidence = predict_image(sample_image)

print("Prediction :", prediction)
print(f"Confidence : {confidence*100:.2f}%")

Prediction : NORMAL
Confidence : 93.09%


In [18]:
# ==========================================
# Create src Directory
# ==========================================

SRC_DIR = PROJECT_DIR / "src"
SRC_DIR.mkdir(exist_ok=True)

print("Source directory:", SRC_DIR)

Source directory: /content/drive/MyDrive/pneumonia_project/src


In [19]:
model_code = """
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

def load_model(model_path, device):

    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    num_features = model.fc.in_features

    model.fc = nn.Linear(num_features, 2)

    model.load_state_dict(
        torch.load(model_path, map_location=device)
    )

    model.to(device)
    model.eval()

    return model
"""

with open(SRC_DIR / "model.py", "w") as f:
    f.write(model_code)

print("✅ model.py created successfully!")

✅ model.py created successfully!


In [20]:
for file in SRC_DIR.iterdir():
    print(file.name)

model.py


In [21]:
utils_code = """
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms

IMG_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

label_map = {
    0: "NORMAL",
    1: "PNEUMONIA"
}

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

def predict_image(model, image, device):

    if isinstance(image, str):
        image = Image.open(image).convert("RGB")
    else:
        image = image.convert("RGB")

    image_tensor = transform(image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = F.softmax(outputs, dim=1)

        confidence, prediction = torch.max(probabilities, dim=1)

    prediction = prediction.item()
    confidence = confidence.item()

    return (
        label_map[prediction],
        confidence
    )
"""

In [22]:
with open(SRC_DIR / "utils.py", "w") as f:
    f.write(utils_code)

print("✅ utils.py created successfully!")

✅ utils.py created successfully!


In [23]:
for file in SRC_DIR.iterdir():
    print(file.name)

model.py
utils.py


In [30]:
gradcam_code = """
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt


class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.target_layer.register_forward_hook(
            self.forward_hook
        )

        self.target_layer.register_full_backward_hook(
            self.backward_hook
        )

    def forward_hook(self, module, input, output):
        self.activations = output

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor):

        self.model.zero_grad()

        output = self.model(input_tensor)

        predicted_class = output.argmax(dim=1)

        output[:, predicted_class].backward()

        gradients = self.gradients[0]
        activations = self.activations[0]

        weights = gradients.mean(dim=(1, 2))

        cam = torch.zeros(
            activations.shape[1:],
            device=activations.device
        )

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = torch.relu(cam)

        cam = cam.detach().cpu().numpy()

        cam = cv2.resize(cam, (224, 224))

        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        return cam


def create_overlay(original, heatmap):

    original = np.array(original) / 255.0

    colored_heatmap = plt.cm.jet(heatmap)[:, :, :3]

    overlay = (
        0.6 * original +
        0.4 * colored_heatmap
    )

    overlay = np.clip(overlay, 0, 1)

    return colored_heatmap, overlay
"""

In [31]:
with open(SRC_DIR / "gradcam.py", "w") as f:
    f.write(gradcam_code)

print("✅ gradcam.py created successfully!")

✅ gradcam.py created successfully!


In [32]:
for file in SRC_DIR.iterdir():
    print(file.name)

model.py
utils.py
gradcam.py


In [34]:
app_code = """
import torch
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st

from pathlib import Path
from PIL import Image

from torchvision.models import resnet18

from src.model import load_model
from src.utils import predict_image, transform
from src.gradcam import GradCAM, create_overlay

# ==========================================
# Configuration
# ==========================================

st.set_page_config(
    page_title="Explainable Pneumonia Detection",
    page_icon="🩺",
    layout="wide"
)

st.title("🩺 Explainable Pneumonia Detection from Chest X-Rays")

st.write(
    "Upload a chest X-ray image to classify it as **NORMAL** or **PNEUMONIA** and visualize the model's attention using **Grad-CAM**."
)

# ==========================================
# Device
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ==========================================
# Paths
# ==========================================

PROJECT_DIR = Path(__file__).parent

MODEL_PATH = PROJECT_DIR / "models" / "best_model.pth"

# ==========================================
# Load Model
# ==========================================

model = load_model(
    MODEL_PATH,
    device
)

grad_cam = GradCAM(
    model=model,
    target_layer=model.layer4
)

# ==========================================
# Upload Image
# ==========================================

uploaded_file = st.file_uploader(
    "Upload Chest X-ray",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file).convert("RGB")

    prediction, confidence = predict_image(
        model,
        image,
        device
    )

    input_tensor = transform(image).unsqueeze(0).to(device)

    heatmap = grad_cam.generate(input_tensor)

    colored_heatmap, overlay = create_overlay(
        image.resize((224,224)),
        heatmap
    )

    st.success(
        f"Prediction: {prediction}"
    )

    st.info(
        f"Confidence: {confidence*100:.2f}%"
    )

    col1, col2, col3 = st.columns(3)

    with col1:
        st.image(
            image,
            caption="Original Image",
            use_container_width=True
        )

    with col2:
        st.image(
            colored_heatmap,
            caption="Grad-CAM Heatmap",
            use_container_width=True
        )

    with col3:
        st.image(
            overlay,
            caption="Overlay",
            use_container_width=True
        )
"""

In [35]:
with open(PROJECT_DIR / "app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully!")

✅ app.py created successfully!


In [36]:
import os

print(os.listdir(PROJECT_DIR))

['chest-xray-pneumonia.zip', 'data', 'notebooks', 'models', 'outputs', 'api', 'src', 'data_splits', 'app.py']


In [37]:
requirements = """
streamlit
torch
torchvision
numpy
pandas
matplotlib
opencv-python
Pillow
scikit-learn
"""

with open(PROJECT_DIR / "requirements.txt", "w") as f:
    f.write(requirements.strip())

print("✅ requirements.txt created successfully!")

✅ requirements.txt created successfully!


In [38]:
print((PROJECT_DIR / "requirements.txt").exists())

True


In [39]:
gitignore = """
# Python
__pycache__/
*.pyc
*.pyo

# Jupyter
.ipynb_checkpoints/

# Streamlit
.streamlit/

# OS files
.DS_Store
Thumbs.db

# Virtual environments
venv/
env/

# Model checkpoints (optional)
*.pth
"""

with open(PROJECT_DIR / ".gitignore", "w") as f:
    f.write(gitignore.strip())

print("✅ .gitignore created successfully!")

✅ .gitignore created successfully!


In [40]:
gitignore = """
# Python
__pycache__/
*.pyc
*.pyo

# Jupyter
.ipynb_checkpoints/

# Streamlit
.streamlit/

# OS files
.DS_Store
Thumbs.db

# Virtual environments
venv/
env/
"""

with open(PROJECT_DIR / ".gitignore", "w") as f:
    f.write(gitignore.strip())

print("✅ .gitignore created successfully!")

✅ .gitignore created successfully!


In [41]:
print((PROJECT_DIR / ".gitignore").exists())

True
